Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1) Storage Method Justification

### Method Chosen: InfluxDB (Time-Series Database)

Justification: Every update such as sensor or state is written as its own InfluxDB measurement, named after the variable itself (e.g. `motor_temperature`, `operational_status`), tagged with `twin_id` and `data_type` (sensor/state). This keeps the schema generic: a new variable can be added to the digital twin at any time without changing the database structure, since every point only ever needs `{measurement_name, twin_id, data_type, value}`. Because InfluxDB fields must be a single type, numeric values are stored under `value_num` and text values under `value_str`, so both sensor readings and state labels fit the same write path.

This is a better fit than a flat file (no querying by time or by twin) or a fixed-schema relational table (would need a new column for every new variable). InfluxDB's tags let us filter by twin and variable type, and its time-indexing lets us pull either the latest value or the full history from the same store.

## 2) Digital Twin State Variables & Sensor Data Justification

For the twin `robot_001`:

**State variables** (`data_type = "state"`)
- `operational_status` (idle / moving / charging) — the robot's current activity, used to decide what it should do next without re-deriving it from raw sensor history.
- `current_zone` (e.g. Zone_A / Zone_B) — the robot's current location context, used for zone-based rules (e.g. restricting speed in certain zones).

**Sensor stream** (`data_type = "sensor"`)
- `motor_temperature` — continuous numeric reading tracked over time to detect overheating trends.
- `battery_voltage` — continuous numeric reading tracked over time to monitor power health and degradation.

State variables describe the twin's *current condition* and change through discrete transitions, so only the latest value usually matters. Sensor variables are continuous numeric readings whose value is meaningful mainly in the context of their history, hence storing both as time-series points, but querying them differently (`last()` for state, a time range for sensors).

## 3) Kafka Stream Manager Justification & Routing

Justification: Kafka sits between the twin (producer) and InfluxDB (consumer/writer), so the producer only needs to know the topic, not the database schema or connection. The consumer can be changed, restarted, or scaled independently, and other consumers (e.g. an alerting service) could subscribe to the same topic later without touching the producer.

Protocol Route/Interface: A local Kafka broker at `127.0.0.1:9092`, topic `digitaltwin.telemetry`. Each message is a generic JSON envelope  `{twin_id, type, measurement_name, value}`  published by the producer and read by a background consumer thread, which writes it into InfluxDB as the measurement named in `measurement_name`, tagged by `twin_id` and `type`. Because the message format is generic rather than hardcoded per-variable, new sensors or state variables can be streamed through the same pipeline with no changes to the producer or consumer code.

In [1]:
%pip install influxdb-client kafka-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached influxdb_client-1.50.0-py3-none-any.whl.metadata (65 kB)
Using cached influxdb_client-1.50.0-py3-none-any.whl (746 kB)
Note: you may need to restart the kernel to use updated packages.


## 4) Connect to the InfluxDB store

In [1]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import json
import time
import threading
from kafka import KafkaProducer, KafkaConsumer

INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "fJkA8iV64KKnTXkuuzLGHLpVBIDXBAWGlm3eyZFteWjECM1Bc89nx_5ikhNerb5fg_hjAyZ6CuVo4OMAzSAJTw==" # 
INFLUX_ORG = "raziq"
INFLUX_BUCKET = "digitaltwin"

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)
query_api = client.query_api()

buckets_api = client.buckets_api()
if not buckets_api.find_bucket_by_name(INFLUX_BUCKET):
    buckets_api.create_bucket(bucket_name=INFLUX_BUCKET, org=INFLUX_ORG)
    print(f"Created bucket '{INFLUX_BUCKET}'")
else:
    print(f"Bucket '{INFLUX_BUCKET}' already exists")

print("InfluxDB reachable:", client.ping())

Bucket 'digitaltwin' already exists
InfluxDB reachable: True


## 7) Define the Kafka Consumer and the parsing logic to write to InfluxDB

In [2]:
KAFKA_BROKER = "127.0.0.1:9092" 
KAFKA_TOPIC = "digitaltwin.telemetry"
consumer_ready = threading.Event()

def write_message_to_influx(msg):
    point = Point(msg["measurement_name"]) \
        .tag("twin_id", msg["twin_id"]) \
        .tag("data_type", msg["type"]) 
    
    if isinstance(msg["value"], (int, float)):
        point.field("value_num", float(msg["value"]))
    else:
        point.field("value_str", str(msg["value"]))
        
    write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=point)

def consume_loop():
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        auto_offset_reset="latest", 
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        group_id=f"digitaltwin-storage-writer-{int(time.time())}"
    )

    consumer.poll(timeout_ms=1000)
    consumer_ready.set()
    
    while True:
        try:
            for message in consumer:
                msg = message.value
                try:
                    write_message_to_influx(msg)
                    print(f"[KAFKA CONSUMER] Stored {msg['type']} '{msg['measurement_name']}' for {msg['twin_id']} in InfluxDB")
                except Exception as e:
                    print(f"[KAFKA CONSUMER ERROR] Failed to write {msg}: {e}")
        except Exception as e:
            print(f"[KAFKA CONSUMER FATAL] Consumer loop crashed, restarting: {e}")
            time.sleep(1)

consumer_thread = threading.Thread(target=consume_loop, daemon=True)
consumer_thread.start()

if consumer_ready.wait(timeout=10):
    print("Kafka consumer subscribed and ready.")
else:
    print("Warning: consumer did not confirm readiness in time.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16060\1765403860.py:18: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Kafka consumer subscribed and ready.


## Step 3: Produce simulated twin data to Kafka

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

digital_twin_updates = [
    # Sensor Streams
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "motor_temperature", "value": 68.5},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "battery_voltage", "value": 24.1},
    
    # State Variables
    {"twin_id": "robot_001", "type": "state", "measurement_name": "operational_status", "value": "moving"},
    {"twin_id": "robot_001", "type": "state", "measurement_name": "current_zone", "value": "Zone_B"}
]

print("Publishing data to Kafka...")
for update in digital_twin_updates:
    producer.send(KAFKA_TOPIC, update)
producer.flush()
print("Data published successfully. Waiting for consumer to process...")

time.sleep(3)

Publishing data to Kafka...
Data published successfully. Waiting for consumer to process...
[KAFKA CONSUMER] Stored sensor 'motor_temperature' for robot_001 in InfluxDB
[KAFKA CONSUMER] Stored sensor 'battery_voltage' for robot_001 in InfluxDB
[KAFKA CONSUMER] Stored state 'operational_status' for robot_001 in InfluxDB
[KAFKA CONSUMER] Stored state 'current_zone' for robot_001 in InfluxDB


## Step 4: Retrieve and validate data from InfluxDB


In [8]:
print(f"--- Retrieving Digital Twin Data from InfluxDB ---")

flux_query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -5m)
  |> filter(fn: (r) => r["twin_id"] == "robot_001")
  |> keep(columns: ["_time", "_measurement", "data_type", "_field", "_value"])
  |> sort(columns: ["_time"], desc: true)
'''

result = query_api.query(org=INFLUX_ORG, query=flux_query)

if not result:
    print("No data found. Ensure Kafka and InfluxDB are running and properly connected.")
else:
    for table in result:
        for record in table.records:
            time_stamp = record.get_time().strftime('%Y-%m-%d %H:%M:%S')
            data_type = record.values.get("data_type")
            measurement = record.values.get("_measurement") 
            value = record.get_value()
            
            print(f"[{time_stamp}] Type: {str(data_type).upper().ljust(7)} | Variable: {str(measurement).ljust(20)} | Value: {value}")

--- Retrieving Digital Twin Data from InfluxDB ---
[2026-07-04 05:08:40] Type: SENSOR  | Variable: battery_voltage      | Value: 24.1
[2026-07-04 05:08:40] Type: SENSOR  | Variable: motor_temperature    | Value: 68.5
[2026-07-04 05:08:40] Type: STATE   | Variable: current_zone         | Value: Zone_B
[2026-07-04 05:08:40] Type: STATE   | Variable: operational_status   | Value: moving
